In [53]:
import pandas as pd
import numpy as np

# =====================================================
# Load Elo ratings
# =====================================================

url = "https://www.eloratings.net/World.tsv"

columns = [
    "rank",
    "previous_rank",
    "country_code",
    "elo",
    "elo_rank_change",
    "max_elo",
    "max_rank",
    "min_elo_since",
    "min_rank_since",
    "lowest_elo",
    "change1",
    "value1",
    "change2",
    "value2",
    "change3",
    "value3",
    "change4",
    "value4",
    "change5",
    "value5",
    "change6",
    "value6",
    "matches",
    "wins",
    "draws",
    "losses",
    "home_matches",
    "home_wins",
    "home_draws",
    "goals_for",
    "goals_against",
]

df = pd.read_csv(url, sep="\t", names=columns)

# =====================================================
# Drop unwanted columns
# =====================================================

drop_cols = [
    "rank",
    "elo_rank_change",
    "max_rank",
    "min_elo_since",
    "min_rank_since",
    "lowest_elo",
    "change1",
    "change2",
    "change3",
    "change4",
    "change5",
    "change6",
]

df.drop(columns=drop_cols, inplace=True)

# =====================================================
# Clean recent form columns
# =====================================================

form_cols = ["value1", "value2", "value3", "value4", "value5", "value6"]

for col in form_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace("−", "-", regex=False)
        .replace("-", np.nan)
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Recent form (sum of last 6 Elo changes)
df["recent_form"] = df[form_cols].sum(axis=1)

# Remove original form columns
df.drop(columns=form_cols, inplace=True)

# =====================================================
# Load team names
# =====================================================

teams_url = "https://www.eloratings.net/en.teams.tsv?_=1781977931498"

teams = pd.read_csv(
    "https://www.eloratings.net/en.teams.tsv?_=1781977931498",
    sep="\t",
    header=None,
    usecols=[0, 1],
    names=["country_code", "team_name"],
    engine="python",
)

df = df.merge(teams, on="country_code", how="left")

# Optional: move team_name to the front
cols = ["team_name"] + [c for c in df.columns if c != "team_name"]
df = df[cols]

# =====================================================
# Preview
# =====================================================

df.head()

,team_name,previous_rank,country_code,elo,max_elo,matches,wins,draws,losses,home_matches,home_wins,home_draws,goals_for,goals_against,recent_form
0,Spain,1,ES,2129,2189,783,341,302,140,462,138,183,1595,699,178.0
1,Argentina,2,AR,2128,2172,1112,381,419,312,613,228,271,2120,1136,173.0
2,France,3,FR,2084,2135,941,475,341,125,476,270,195,1713,1276,180.0
3,England,4,EN,2055,2213,1163,509,524,130,686,215,262,2727,1120,360.0
4,Colombia,5,CO,1998,2069,651,185,229,237,268,205,178,850,743,168.0


In [55]:
df.to_csv("../../data/raw/elo_ratings.csv", index=False)